# Classifying Strains by Function

Same real methodology as the bioinformatics course's notebook 05 — Lasso
logistic regression, cross-validated, scored by balanced accuracy — now
predicting **pathogenic vs. commensal** from strain-level gene presence
instead of species-level presence.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

df = pd.read_csv("ecoli_strain_profiles.csv")
virulence_genes = ["stx1", "stx2", "eae", "papC", "hlyA"]   # blaKPC excluded on purpose — see below

X = df[virulence_genes].values
y = df["pathogenic"].values
print(X.shape, y.shape, "->", y.sum(), "pathogenic /", len(y) - y.sum(), "commensal")

Notice `blaKPC` is deliberately left out of `X`. Notebooks 05 and 06
established it's a roughly independent trait — including a
resistance-only gene when predicting *virulence* specifically would add
noise, not signal. Choosing which columns belong in a model is itself a
real modeling decision, not just a mechanical step.

## 2. Train and inspect weights

In [ ]:
model = LogisticRegression(penalty="l1", solver="liblinear", C=0.5)
model.fit(X, y)
weights = pd.Series(model.coef_[0], index=virulence_genes).sort_values()
print(weights.round(3))

## 3. Cross-validate honestly

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")
print("fold scores:", scores.round(3))
print(f"mean balanced accuracy: {scores.mean():.3f}")

### EXPLAIN #1
*Run the cell above and read the actual fold scores, not just the mean —
notice how much they vary fold to fold. With 89 strains split 5 ways,
each fold tests fewer than 20 strains. Connect this to the
bioinformatics course's lesson on cross-validation and small samples:
does a wide spread across folds mean the model is bad, or does it mean
this dataset is small enough that any single fold's score is a noisy
estimate?*

> your answer here

### 🔧 YOUR TURN #1
Add `"blaKPC"` back into `virulence_genes` and re-run training and
cross-validation. Does mean balanced accuracy meaningfully improve, get
worse, or stay about the same — consistent with notebook 05's claim that
it's an independent trait?

In [ ]:
# Your code here

## Done

**Next:** `10_what_strain_data_still_cant_tell_you.ipynb`.